# 세 구조로 같은 주문을 계산해 봐요

카페라떼 2잔에 학생 할인을 적용해요. A·B·C의 **최종 금액이 모두 7,200원인지** 확인하고, 할인율을 바꿀 위치를 찾아요. 영수증 모양이나 잘못된 입력을 처리하는 방식까지 같은 것은 아니에요.

실행 1과 실행 2는 **이 노트북 하나에서 이어서** 해요.

## 시작하기

1. 위 메뉴에서 **파일 → Drive에 사본 저장**을 눌러요. 내 사본에 답을 남길 수 있어요.
2. 첫 코드 칸 왼쪽의 **▶**를 눌러요. `준비 완료`가 나오면 다음 칸으로 가요.
3. 수업 중에는 안내한 칸만 실행해요. **Shift+Enter**도 같은 칸을 실행하는 방법이에요.

`Cell`은 코드나 설명이 들어 있는 칸이에요. 런타임이 초기화되어 파일이나 변수가 사라졌다면 첫 칸부터 다시 실행해요. 단순히 브라우저를 다시 여는 것과는 달라요.

코드 앞의 `#`는 설명이에요. 실행되지 않아요. `import`는 다른 파일의 이름을 가져오고, 줄 앞의 `!`는 Python 대신 터미널 명령을 실행해요.


## 수정본으로 바꾸려면

GitHub 원본이 바뀌어도 이미 만든 Drive 사본은 자동으로 바뀌지 않아요. 적어 둔 답과 코드를 먼저 저장하세요. [최신 원본 열기](https://colab.research.google.com/github/GoBeromsu/jnu-llmops-precourse-day3/blob/main/notebooks/day3_structures.ipynb)에서 새 Drive 사본을 만든 뒤, 내 답과 수정한 조건만 옮기고 첫 코드 칸부터 실행하세요. 기존 사본을 지우거나 덮어쓰지 않아도 돼요.


In [ ]:
# 필요한 파일을 Colab으로 받아 와요. 실행 상태가 초기화되면 다시 준비해요.
# import는 이름을 가져오고, !는 터미널 명령을 실행하는 표시예요.
import os, sys

if not os.path.isdir("jnu-llmops-precourse-day2"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 내려받기가 실패했는데 준비됐다고 표시하지 않아요.
for filename in (
    "jnu-llmops-precourse-day2/solution/catalog.py",
    "jnu-llmops-precourse-day2/solution/order.py",
    "jnu-llmops-precourse-day2/solution/pricing.py",
    "jnu-llmops-precourse-day2/solution/receipt.py",
    "jnu-llmops-precourse-day3/data/orders.json",
    "jnu-llmops-precourse-day3/data/expected_day3.json",
    "jnu-llmops-precourse-day3/data/expected_day4.json",
):
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"준비 파일이 없어요: {filename}. 위 다운로드 오류를 강사에게 보여 주세요.")

sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")


## 버전 A · 어제 만든 파일을 불러요

다음 칸을 실행하기 전에 최종 금액을 예상해 보세요. 실행하면 영수증이 나와요.

`Order`는 주문을 담고, `calculate_bill`은 금액을 계산해요. `format_receipt`는 영수증 글자를 만들어요. `from 파일이름 import 이름`은 그 파일의 이름을 여기서 쓰겠다는 뜻이에요.

**A는 어제 완료본을 그대로 가져와요.** 메뉴는 `catalog`, 주문은 `order`, 계산은 `pricing`, 영수증은 `receipt`에 있어요. `Order` 안에는 `OrderItem` 객체를 담아요. 수량은 추가할 때와 계산할 때 검사해요. 오늘은 그 파일들을 다시 쓰지 않고 `import`로 불러요.


In [ ]:
# 어제 완료본의 네 파일에서 이름 넷을 꺼내 옵니다.
# from 파일이름 import 이름  = "그 파일 안의 이 이름을 여기서 쓰겠다"
from catalog import MENU               # 메뉴판 (이름 → 가격)
from order import Order                # 주문 상자를 만드는 틀
from pricing import calculate_bill     # 주문 → 합계·할인·최종 금액
from receipt import format_receipt     # 주문 + 금액 → 영수증 문자열

order_a = Order()                                    # 빈 주문 하나
order_a.add("카페라떼", 2, MENU["카페라떼"])          # 항목 추가 (메뉴, 수량, 단가)
bill_a = calculate_bill(order_a, True)               # True = 학생 할인
print(format_receipt(order_a, bill_a))               # 영수증을 화면에 찍는다
total_a = bill_a.total                               # 최종 금액을 이름에 남겨 둔다

## 버전 B · 항목을 묶음으로 담아요

다음 칸은 `(메뉴, 수량, 단가)`를 한 묶음으로 저장해요. 계산은 이 칸에서 해요. 실행하면 수량과 최종 금액이 나와요.

**A에서 무엇이 달라졌나요?** `OrderItem` 대신 `(메뉴, 수량, 단가)` tuple을 담아요. 단가는 직접 넣고 계산은 노트북으로 꺼냈어요. 별도 영수증 함수 없이 결과를 출력해요. 단순화하면서 **입력 검사도 빠졌어요.** 따라서 모든 동작이 A와 같은 코드는 아니에요.


In [ ]:
class OrderB:
    def __init__(self):
        self.items = []                      # (메뉴, 수량, 단가)
    def add(self, menu_name, quantity, unit_price):
        self.items.append((menu_name, quantity, unit_price))

order_b = OrderB()
order_b.add("카페라떼", 2, 4000)
subtotal_b = sum(q * p for _, q, p in order_b.items)
total_b = subtotal_b - subtotal_b * 10 // 100      # 학생 할인 10%
print("수량:", order_b.items[0][1])
print("최종 금액:", total_b, "원")

## 버전 C · Cafe 하나에 담아요

메뉴판·주문·계산·영수증을 한 Class에 넣었어요. 실행한 뒤 A·B와 최종 금액을 비교해 보세요.

**B에서 무엇이 달라졌나요?** 메뉴판과 계산·영수증을 `Cafe` 안으로 모았어요. 주문에는 `(메뉴, 수량)`만 담고, 단가는 안의 메뉴판에서 찾아요. 하지만 **수량 검사는 여전히 없어요.** 한 Class이기 때문에 검사할 수 없는 것은 아니에요.


In [ ]:
class Cafe:
    MENU = {"아메리카노": 3000, "카페라떼": 4000, "초코라떼": 4500}
    def __init__(self):
        self.items = []                      # (메뉴, 수량)
    def add(self, menu_name, quantity):
        self.items.append((menu_name, quantity))
    def total(self, is_student):
        subtotal = sum(q * self.MENU[m] for m, q in self.items)
        discount = subtotal * 10 // 100 if is_student else 0
        return subtotal - discount
    def receipt(self, is_student):
        lines = [f"{m} x {q}" for m, q in self.items]
        lines.append(f"최종 금액: {self.total(is_student)}원")
        return "\n".join(lines)

cafe = Cafe()
cafe.add("카페라떼", 2)
print(cafe.receipt(True))
total_c = cafe.total(True)

## 세 금액을 비교해요

다음 칸을 실행하면 A·B·C의 금액이 나란히 나와요. 모두 7,200원이면 검사를 통과해요.

`AssertionError`가 나오면 세 금액 중 다른 값을 먼저 찾으세요. 그 버전의 입력과 계산 줄을 확인한 뒤 다시 실행해요.

| 비교 | A | B | C |
| --- | --- | --- | --- |
| 주문 항목 | Order 안의 OrderItem | OrderB 안의 tuple | Cafe 안의 tuple |
| 계산 위치 | pricing 파일 | 노트북 코드 칸 | Cafe.total |
| 영수증 | receipt 파일 | 결과를 바로 출력 | Cafe.receipt |
| 수량 검사 | 추가·계산 시 검사 | 없음 | 없음 |

여기서 같은 것은 **카페라떼2잔 학생 주문의 최종 금액**이에요. 모든 입력에서 같은 동작이라는 뜻은 아니에요. 다음 칸에서 수량0을 확인해요.


In [ ]:
print("A 네 겹   :", total_a)
print("B 두 겹   :", total_b)
print("C 한 덩어리:", total_c)
assert total_a == total_b == total_c == 7200
print("세 구조, 같은 결과")

## 수량0으로 차이를 확인해요

A·B·C의 원래 수량2는 그대로 두세요. 이미0으로 바꿨다면2로 돌리고 해당 칸부터 다시 실행하세요. 아래 칸이 **새 주문을 만들어** 세 버전에 수량0을 넣어요. 이전 실행의 금액을 잘못 읽지 않기 위한 칸이에요.

실행 전 예상하세요. A·B·C가 모두 거부할까요, 아니면0원을 계산하는 버전도 있을까요? 이 칸은 오류를 글자로 보여 주고 다음 버전도 확인해요.


In [ ]:
zero_results = {}
for version in ("A", "B", "C"):
    try:
        if version == "A":
            zero_a = Order()
            zero_a.add("카페라떼", 0, 4000)
            zero_total = calculate_bill(zero_a, True).total
        elif version == "B":
            zero_b = OrderB()
            zero_b.add("카페라떼", 0, 4000)
            zero_subtotal = sum(q * p for _, q, p in zero_b.items)
            zero_total = zero_subtotal - zero_subtotal * 10 // 100
        else:
            zero_c = Cafe()
            zero_c.add("카페라떼", 0)
            zero_total = zero_c.total(True)
        zero_results[version] = zero_total
    except (TypeError, ValueError) as error:
        zero_results[version] = f"{type(error).__name__}: {error}"
    print(version, "→", zero_results[version])


## 왜 다르고, 왜 모듈로 나누나요?

**직접 원인은 검사 유무예요.** A의 `Order.add`는 수량이1~10 밖이면 `ValueError`로 거부해요. B와C에는 그 검사가 없어서 `0 × 가격`을 계산해요. 파일이 많아서 A가 안전해진 것도, 한 Class이라 C가 검사하지 못하는 것도 아니에요.

그렇다면 B와C에 검사 코드를 각각 복사하면 될까요? 가능하지만 허용 수량을 바꿀 때 여러 곳을 고쳐야 해요. 한 곳을 빠뜨리면 같은 주문을 다르게 처리할 수 있어요.

**함께 쓰는 기능을 모아 불러 쓰면 재사용과 변경이 쉬워져요.** 어제 `pricing.py`의 `calc_total`은 수량·단가를 검사하고 항목 금액을 계산해요. A의 전체 계산도 이 함수를 사용해요. 아래에서는 같은 함수를 노트북에서 직접 불러요.

8,000원은 **할인 전 항목 금액**이에요. 앞의7,200원은 학생 할인 후 금액이므로 구분하세요.


In [ ]:
from pricing import calc_total

amount = calc_total(2, 4000)
print("할인 전:", amount, "원")

try:
    calc_total(0, 4000)
except (TypeError, ValueError) as error:
    print("공통 함수의 수량0 검사:", type(error).__name__, str(error))


## B와C에서도 같은 계산을 불러 쓸 수 있어요

아래는 계산 줄을 공통 함수로 바꾸는 **설명 예시**예요. 지금 B와C가 자동으로 바뀐 것은 아니에요.

```python
# B의 계산 줄을 이렇게 바꿀 수 있어요.
subtotal_b = sum(calc_total(q, p) for _, q, p in order_b.items)

# C의 total 안에서도 같은 함수를 불러 쓸 수 있어요.
subtotal = sum(calc_total(q, self.MENU[m]) for m, q in self.items)
```

파일만 늘리는 것이 목적은 아니에요. **어떤 기능을 함께 쓰고, 어디에서 고칠지 정하는 것**이 목적이에요. A에도 주문 추가와 계산 단계의 검사가 각각 있으므로, 이미 모든 검사가 한곳에 모였다는 뜻은 아니에요.


## 찾은 위치를 적어요

이 설명 칸을 두 번 누르면 답을 적을 수 있어요. 먼저 A·B·C에서 무엇이 달라졌는지 설명한 뒤, 아래 질문에 답해 보세요.

1. 학생 할인을 10%에서 15%로 바꾸려면 A·B·C의 어느 줄을 고치나요?
2. 수량0에서 A는 왜 멈추고 B·C는 왜0원을 계산했나요? 파일 개수 때문인가요, 검사 때문인가요?
3. 계산만 따로 확인하고 싶다면 어느 버전이 편한가요? 이유는 무엇인가요?

4. 같은 검사·계산을 여러 곳에서 써야 한다면 어느 기능을 함께 불러 쓰겠나요?

내 답:
- A에서 가져온 것 / B에서 뺀 것 / C로 모은 것:
- 할인율을 바꿀 위치: A / B / C
- 수량 0을 넣은 결과:
- 따로 확인하기 편한 버전과 이유:
- 다시 쓸 기능과 그 기능을 고칠 위치:

코드를 아직 다 읽지 못해도 괜찮아요. 찾은 줄을 먼저 적고, 남은 질문을 강사에게 보여 주세요.